In [ ]:
import cv2
from pathlib import Path

In [7]:
video_path = Path("test_converted_videos/10.0.16.2_202307070748.mp4")
output_folder = "active_frames"

In [8]:
from collections import deque
cap = cv2.VideoCapture(video_path)
prev_frame = None
saved_count = 0
frame_index = 0

'''while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_index += 1

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21,21), 0)

    if prev_frame is not None:
        frame_delta = cv2.absdiff(prev_frame, gray)
        thresh = cv2.threshold(frame_delta, 25, 255, cv2.THRESH_BINARY)[1]'''
frame_buffer = deque(maxlen=3)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_index += 1
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = clahe.apply(gray)
        gray = cv2.GaussianBlur(gray, (5,5), 0)

        # threshold 
        frame_buffer.append(gray)
        accumulated = gray.copy()
        for old_frame in frame_buffer:
            accumulated = cv2.max(accumulated, old_frame)

        _, thresh = cv2.threshold(accumulated, 15 , 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        if cv2.countNonZero(thresh) > 500:
            cv2.imwrite(f"{output_folder}/frame_{frame_index:05d}.jpg", frame)
            saved_count += 1
    

cap.release()
print(f"Saved {saved_count} active frames.")

Saved 168 active frames.
